# Libraries

# Aux functions

In [ ]:
from collections.abc import Sequence

import matplotlib.pyplot as plt
import pandas as pd


def check_nans(df: pd.DataFrame, columns: Sequence[str],
                show_percent: bool = True,  # noqa: FBT001, FBT002
                sort: bool = True) -> pd.DataFrame:  # noqa: FBT001, FBT002
    """
    Quick check for NaNs in dataframe columns.
    Args:
        df: pandas.DataFrame
        columns: list of column names to check (None -> all columns)
        show_percent: include missing percentage column
        sort: sort result by missing_count desc
    Returns:
        pandas.DataFrame with index = column and columns:
            missing_count, missing_pct (optional), non_missing_count, dtype
    """  # noqa: DOC201

    if columns is None:  # type: ignore  # noqa: PGH003
        columns = list(df.columns)  # two spaces before comment
    total = len(df)  # two spaces before comment
    rows = []
    for col in columns:
        ser = df[col]  # two spaces before comment
        missing = int(ser.isna().sum())  # two spaces before comment

        rows.append({  # type: ignore  # noqa: PGH003
            "column": col,
            "missing_count": missing,
            "missing_pct": (missing / total) * 100 if total > 0 else 0.0,
            "non_missing_count": total - missing,
            "dtype": str(ser.dtype)
        })
    out = pd.DataFrame(rows).set_index("column")
    if not show_percent:
        out = out.drop(columns="missing_pct")
    if sort:
        out = out.sort_values("missing_count", ascending=False)
    return out

In [ ]:
def plot_value_counts(df: pd.DataFrame, column_name: str, xlabel: str | None = None, ylabel: str = "Frequency") -> None:  # type: ignore  # noqa: E501, PGH003
    """
    Plots a bar chart of value counts for a specified column in a DataFrame.

    Parameters:
        df (pd.DataFrame): The DataFrame containing the data.
        column_name (str): The column name to plot value counts for.
        xlabel (str | None, optional): Label for the x-axis. Defaults to the column name
        ylabel (str, optional): Label for the y-axis. Defaults to 'Frequency'.
    """
    counts = df[column_name].value_counts().sort_index()

    plt.figure(figsize=(10, 6))  # type: ignore  # noqa: PGH003
    bars = counts.plot(kind="bar", edgecolor="black")

    # Add value labels on top of each bar
    for container in bars.containers:
        bars.bar_label(container, fmt="%d")  # type: ignore  # noqa: PGH003

    plt.xlabel(xlabel or column_name)  # type: ignore  # noqa: PGH003
    plt.ylabel(ylabel)  # type: ignore  # noqa: PGH003
    plt.xticks(rotation=0)  # type: ignore  # noqa: PGH003
    plt.grid(axis="y", alpha=0.3)  # type: ignore  # noqa: PGH003
    plt.tight_layout()
    plt.show()  # type: ignore  # noqa: PGH003

# Data Loading

## Configuration

This notebook processes **two NHANES cycles** with unified feature engineering:

- **2021-2023 cycle** (`merged_data.csv`)
- **2017-2020 cycle** (`merged_data2017.csv`)

Both cycles share most features, but differ in physical activity variables:
- 2021-2023: `PAD790Q/U/800`, `PAD810Q/U/820` (frequency + unit + duration)
- 2017-2020: `PAQ650/655/660`, `PAQ665/670/675` (gate + days/week + duration)

**Strategy**: Load both, apply identical transformations, handle PA differences, concatenate final output.

In [ ]:
# Load both NHANES cycles
df_2021 = pd.read_csv("../dataset/merged_data-2021-2023 final.csv", sep=",")  # type: ignore
df_2017 = pd.read_csv("../dataset/merged_data-2017-2020 final.csv", sep=",")  # type: ignore

# Add cycle identifiers for tracking
df_2021["cycle"] = "2021-2023"
df_2017["cycle"] = "2017-2020"

print(f"2021-2023 cycle: {len(df_2021):,} rows, {len(df_2021.columns)} columns")  # noqa: T201
print(f"2017-2020 cycle: {len(df_2017):,} rows, {len(df_2017.columns)} columns")  # noqa: T201

In [ ]:
# Preview both datasets
print("2021-2023 cycle preview:")  # noqa: T201
display(df_2021.head(3))  # type: ignore  # noqa: PGH003

print("\n2017-2020 cycle preview:")  # noqa: T201
display(df_2017.head(3))  # type: ignore  # noqa: PGH003

## Feature filtering

### Cycle-Specific Column Definitions

The two cycles have different physical activity variable names, so we need separate column lists.

In [ ]:
# Common columns across both cycles
COMMON_COLUMNS = [
    # Identifier
    "SEQN",

    # Demographics
    "RIDAGEYR",     # Age
    "RIAGENDR",     # Gender
    "RIDRETH3",     # Race/Ethnicity

    # Socioeconomic Status
    "DMDEDUC2",     # Education level
    "INDFMPIR",     # Income-to-poverty ratio
    "DMDMARTZ",     # Marital status

    # Body Measurements
    "BMXBMI",       # BMI
    "BMXWAIST",     # Waist circumference
    "BMXWT",        # Weight
    "BMXHT",        # Height

    # Blood Pressure (3 readings each)
    "BPXOSY1", "BPXOSY2", "BPXOSY3",  # Systolic readings
    "BPXODI1", "BPXODI2", "BPXODI3",  # Diastolic readings

    # Blood Pressure & Cholesterol Questionnaire (BPQ)
    "BPQ020",       # Ever told you had high blood pressure
    "BPQ080",       # Doctor told you - high cholesterol level

    # Lab: Glycohemoglobin (for lab-defined target)
    "LBXGH",        # Glycohemoglobin (HbA1c) %

    # Smoking
    "SMQ020",       # Ever smoked at least 100 cigarettes
    "SMQ040",       # Current smoking status
    "SMD650",       # Cigarettes per day

    # Alcohol
    "ALQ121",       # Drinking frequency
    "ALQ130",       # Drinks per day
    "ALQ170",       # Binge drinking episodes

    # Sleep
    "SLD012",       # Sleep hours - weekdays (derived, continuous 2-14)
    "SLD013",       # Sleep hours - weekends (derived, continuous 2-14)

    # Depression (PHQ-9)
    "DPQ010",       # Little interest in doing things
    "DPQ020",       # Feeling down, depressed, hopeless
    "DPQ030",       # Trouble sleeping or sleeping too much
    "DPQ040",       # Feeling tired or having little energy
    "DPQ050",       # Poor appetite or overeating
    "DPQ060",       # Feeling bad about yourself
    "DPQ070",       # Trouble concentrating
    "DPQ080",       # Moving or speaking slowly or too fast
    "DPQ090",       # Thoughts of being better off dead

    # Target Variable
    "DIQ010"        # Diabetes diagnosis
]

# 2021-2023 cycle specific columns
CYCLE_2021_ONLY = [
    "DMDHHSIZ",     # Household size (not in 2017-2020)
    "WTMEC2YR",     # Survey weights 2-year (MEC exam weight)
]

# 2017-2020 cycle specific columns
CYCLE_2017_ONLY = [
    "WTMECPRP",     # Pre-pandemic survey weights (2017-2020)
]

# Physical Activity - 2021-2023 cycle
PA_COLUMNS_2021 = [
    "PAD790Q",      # Moderate activity frequency
    "PAD790U",      # Moderate activity unit (D/W/M/Y)
    "PAD800",       # Moderate activity minutes per session
    "PAD810Q",      # Vigorous activity frequency
    "PAD810U",      # Vigorous activity unit (D/W/M/Y)
    "PAD820",       # Vigorous activity minutes per session
    "PAD680",       # Sedentary minutes per day
]

# Physical Activity - 2017-2020 cycle
PA_COLUMNS_2017 = [
    "PAQ650",       # Vigorous recreational activities (Yes/No gate)
    "PAQ655",       # Vigorous: days per week
    "PAD660",       # Vigorous: minutes per session
    "PAQ665",       # Moderate recreational activities (Yes/No gate)
    "PAQ670",       # Moderate: days per week
    "PAD675",       # Moderate: minutes per session
    "PAD680",       # Sedentary minutes per day (same as 2021-2023)
]

# Build cycle-specific required columns
REQUIRED_COLUMNS_2021 = COMMON_COLUMNS + CYCLE_2021_ONLY + PA_COLUMNS_2021
REQUIRED_COLUMNS_2017 = COMMON_COLUMNS + CYCLE_2017_ONLY + PA_COLUMNS_2017

print(f"2021-2023 cycle requires {len(REQUIRED_COLUMNS_2021)} columns")  # noqa: T201
print(f"2017-2020 cycle requires {len(REQUIRED_COLUMNS_2017)} columns")  # noqa: T201

In [ ]:
# Filter to required columns for each cycle
df_2021 = df_2021[[*REQUIRED_COLUMNS_2021, "cycle"]].copy()
df_2017 = df_2017[[*REQUIRED_COLUMNS_2017, "cycle"]].copy()

print("After filtering:")  # noqa: T201
print(f"  2021-2023: {len(df_2021)} rows × {len(df_2021.columns)} columns")  # noqa: T201
print(f"  2017-2020: {len(df_2017)} rows × {len(df_2017.columns)} columns")  # noqa: T201

In [ ]:
# Add placeholder columns for 2017 data to match 2021 schema
# DMDHHSIZ (Household size) doesn't exist in 2017-2020 cycle
if "DMDHHSIZ" not in df_2017.columns:
    df_2017["DMDHHSIZ"] = pd.NA  # type: ignore
    print("✓ Added DMDHHSIZ placeholder (NaN) for 2017-2020 cycle")  # noqa: T201

# Verify both datasets now have consistent schemas
print("\nColumn consistency check:")  # noqa: T201
print(f"  2021-2023 has DMDHHSIZ: {'DMDHHSIZ' in df_2021.columns}")  # noqa: T201
print(f"  2017-2020 has DMDHHSIZ: {'DMDHHSIZ' in df_2017.columns}")  # noqa: T201

Feature groups

In [ ]:
# Demographics (common across both cycles)
DEMO_COLUMNS = {
    "CATEGORICAL": [
        "RIAGENDR",     # Gender
        "RIDRETH3",     # Race/Ethnicity
        "DMDEDUC2",     # Education level
        "DMDMARTZ"      # Marital status
    ],
    "CONTINUOUS": [
        "RIDAGEYR",     # Age
        "INDFMPIR",     # Income-to-poverty ratio
        # Note: DMDHHSIZ (Household size) only in 2021-2023, not in 2017-2020
]}

# Body Measurements
BMX_COLUMNS = [
    "BMXBMI",       # BMI
    "BMXWAIST",     # Waist circumference
    "BMXWT",        # Weight
    "BMXHT"         # Height
]

# Blood Pressure (3 readings each)
BP_COLUMNS = [
    "BPXOSY1", "BPXOSY2", "BPXOSY3",  # Systolic readings
    "BPXODI1", "BPXODI2", "BPXODI3"   # Diastolic readings
]

# Blood Pressure & Cholesterol Questionnaire (Metabolic History)
BPQ_COLUMNS = [
    "BPQ020",       # Ever told high blood pressure
    "BPQ080",       # Ever told high cholesterol
]

# Lab features (for target construction only — NOT model features)
LAB_COLUMNS = [
    "LBXGH",        # Glycohemoglobin (HbA1c) %
]

# Smoking
SMQ_COLUMNS = [
    "SMQ020",       # Ever smoked at least 100 cigarettes
    "SMQ040",       # Current smoking status
    "SMD650"        # Cigarettes per day
]

# Alcohol
ALQ_COLUMNS = [
    "ALQ121",       # Drinking frequency
    "ALQ130",       # Drinks per day
    "ALQ170"        # Binge drinking episodes
]

# Sleep
SLQ_COLUMNS = [
    "SLD012",       # Sleep hours - weekdays
    "SLD013",       # Sleep hours - weekends
]

# Depression (PHQ-9 items)
DPQ_COLUMNS = [
    "DPQ010", "DPQ020", "DPQ030", "DPQ040", "DPQ050",
    "DPQ060", "DPQ070", "DPQ080", "DPQ090",
]

# Physical Activity - will be handled separately per cycle

---
📊 **Original EDA Available in Separate Notebook**

For exploratory data analysis validating the 2021-2023 cycle against NHANES documentation, see:

**[nhanes_eda_2021_2023.ipynb](nhanes_eda_2021_2023.ipynb)**

The cells below are legacy EDA code that reference a single `df` variable (now obsolete). They are preserved here for reference but **will produce errors if run**.

**Skip to "Feature Engineering (Both Cycles)"** section below for the production transformation pipeline.

---

# Feature Engineering

# Feature Engineering (Both Cycles)

Now we'll apply transformations to both datasets. Most transformations are identical, but physical activity requires cycle-specific handling.

## Common Transformations Function

This function applies identical transformations to both cycles:
- Target variable (diabetes/prediabetes)
- Demographics (education, marital status, gender)
- Blood pressure averaging
- Smoking features
- Alcohol features

In [ ]:
def apply_common_transformations(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply common feature engineering transformations to both NHANES cycles.

    Args:
        df: DataFrame with raw NHANES data

    Returns:
        DataFrame with transformed features
    """
    df = df.copy()

    # ========== TARGET VARIABLE ==========
    # Create binary target: diabetes (1) or prediabetes (3) -> 1, no diabetes (2) -> 0
    df["has_diabetes_or_prediabetes"] = df["DIQ010"].map({
        1: 1,  # Yes (Diabetes)
        2: 0,  # No
        3: 1,  # Borderline (Prediabetes)
    })
    # Filter to valid responses only
    df = df[df["has_diabetes_or_prediabetes"].isin([0, 1])].copy()
    df["has_diabetes_or_prediabetes"] = df["has_diabetes_or_prediabetes"].astype(int)
    df = df.drop(columns=["DIQ010"])

    # ========== DEMOGRAPHICS ==========
    # Education level: ordinal encoding (0=Don't know, 1-5=education levels)
    df["education_level"] = df["DMDEDUC2"].replace({9: 0}).copy()  # type: ignore
    df = df.drop(columns=["DMDEDUC2"])

    # Marital status: one-hot encoding (k-1 approach)
    df["has_partner"] = (df["DMDMARTZ"] == 1).astype(int)
    df["had_partner"] = (df["DMDMARTZ"] == 2).astype(int)  # noqa: PLR2004
    df = df.drop(columns=["DMDMARTZ"])

    # Gender: binary encoding
    df["is_female"] = (df["RIAGENDR"] == 2).astype(int)  # noqa: PLR2004
    df = df.drop(columns=["RIAGENDR"])

    # ========== BLOOD PRESSURE ==========
    # Average across 3 readings
    systolic_cols = ["BPXOSY1", "BPXOSY2", "BPXOSY3"]
    diastolic_cols = ["BPXODI1", "BPXODI2", "BPXODI3"]
    df["systolic_bp"] = df[systolic_cols].mean(axis=1)
    df["diastolic_bp"] = df[diastolic_cols].mean(axis=1)
    df = df.drop(columns=systolic_cols + diastolic_cols)

    # ========== METABOLIC HISTORY (BPQ) ==========
    # BPQ020: Ever told high blood pressure -> binary
    # Recode: 1=Yes -> 1, 2=No -> 0, 7=Refused/9=Don't know -> NaN
    df["told_high_bp"] = df["BPQ020"].replace({7: pd.NA, 9: pd.NA}).map({1: 1, 2: 0})
    df = df.drop(columns=["BPQ020"])

    # BPQ080: Ever told high cholesterol -> binary
    # Recode: 1=Yes -> 1, 2=No -> 0, 7=Refused/9=Don't know -> NaN
    df["told_high_cholesterol"] = df["BPQ080"].replace({7: pd.NA, 9: pd.NA}).map({1: 1, 2: 0})
    df = df.drop(columns=["BPQ080"])

    # ========== LAB-DEFINED TARGET (HbA1c) ==========
    # LBXGH: Glycohemoglobin (HbA1c) percentage
    # Used to construct lab_positive target, NOT as a model feature
    # lab_positive = HbA1c >= 5.7% (ADA prediabetes threshold)
    # Note: Fasting glucose (LBXGLU) excluded due to separate subsample weights
    df["lab_positive"] = (df["LBXGH"] >= 5.7).astype("Int64")  # nullable int for NaN preservation
    # NaN in LBXGH -> NaN in lab_positive (participants without lab data)
    df.loc[df["LBXGH"].isna(), "lab_positive"] = pd.NA

    # Undiagnosed: lab_positive AND self_report_negative
    # has_diabetes_or_prediabetes already exists (created above)
    df["undiagnosed"] = (
        (df["lab_positive"] == 1) & (df["has_diabetes_or_prediabetes"] == 0)
    ).astype("Int64")
    # Preserve NaN: if lab_positive is NaN, undiagnosed should be NaN
    df.loc[df["lab_positive"].isna(), "undiagnosed"] = pd.NA

    # Drop raw LBXGH — we only need the derived targets
    df = df.drop(columns=["LBXGH"])

    # ========== SMOKING ==========
    # Clean refused/don't know -> never smokers
    df["SMQ020_clean"] = df["SMQ020"].replace({7: 2, 9: 2})  # type: ignore
    df["ever_smoker"] = df["SMQ020_clean"].map({1: 1, 2: 0})
    df["is_current_smoker"] = (
        (df["SMQ020_clean"] == 1) &
        (df["SMQ040"].isin([1, 2]))  # type: ignore
    ).astype(int)
    df = df.drop(columns=["SMQ020", "SMQ020_clean", "SMQ040", "SMD650"])

    # ========== ALCOHOL ==========
    # Clean refused/don't know -> NaN
    df["ALQ121"] = df["ALQ121"].replace({77: pd.NA, 99: pd.NA})  # type: ignore
    df["ALQ130"] = df["ALQ130"].replace({777: pd.NA, 999: pd.NA})  # type: ignore
    df["ALQ170"] = df["ALQ170"].replace({777: pd.NA, 999: pd.NA})  # type: ignore

    # Group ALQ121 into 5 ordinal levels
    alq121_grouped = df["ALQ121"].map({
        0: 0,   # Never
        1: 1, 2: 1,   # Daily
        3: 2, 4: 2,   # Weekly
        5: 3, 6: 3, 7: 3,   # Monthly
        8: 4, 9: 4, 10: 4   # Occasional
    })
    df["drinking_frequency"] = alq121_grouped.map({
        0: 0,  # Never
        4: 1,  # Occasional
        3: 2,  # Monthly
        2: 3,  # Weekly
        1: 4   # Daily
    })
    df["drinks_per_day"] = pd.to_numeric(df["ALQ130"], errors="coerce")  # type: ignore
    df["binge_episodes_month"] = pd.to_numeric(df["ALQ170"], errors="coerce")  # type: ignore
    df = df.drop(columns=["ALQ121", "ALQ130", "ALQ170"])

    # ========== SLEEP ==========
    # SLD012 and SLD013 are already derived variables (continuous, 2-14 range)
    # No recoding needed — values 2 and 14 are floor/ceiling codes but still numeric
    # Just rename for clarity
    df["sleep_hours_weekday"] = df["SLD012"]
    df["sleep_hours_weekend"] = df["SLD013"]
    df = df.drop(columns=["SLD012", "SLD013"])

    # ========== DEPRESSION (PHQ-9) ==========
    # PHQ-9: 9 items scored 0-3, total score 0-27
    # Recode refused (7) and don't know (9) to NaN
    dpq_items = ["DPQ010", "DPQ020", "DPQ030", "DPQ040", "DPQ050",
                 "DPQ060", "DPQ070", "DPQ080", "DPQ090"]
    for col in dpq_items:
        df[col] = df[col].replace({7: pd.NA, 9: pd.NA})

    # Compute PHQ-9 total score (0-27)
    # min_count=9: returns NaN if ANY item is missing (correct clinical scoring)
    df["phq9_score"] = df[dpq_items].sum(axis=1, min_count=9)
    df = df.drop(columns=dpq_items)

    return df

### Transformation Rationale

The common transformations applied to both cycles include:

#### 1. Target Variable: `has_diabetes_or_prediabetes`
**Source:** `DIQ010` - "Doctor told you have diabetes?"
- **Positive class (1):** Diabetes (code 1) OR Prediabetes (code 3)
- **Negative class (0):** No diabetes (code 2)
- **Excluded:** Refused (7) and Don't know (9)
- **Rationale:** Combines diabetes and prediabetes for conservative screening. Both share the same metabolic pathway (insulin resistance), and ~70% of prediabetes cases progress to diabetes. Both benefit from early intervention.

#### 2. Education Level: `education_level`
**Source:** `DMDEDUC2` (1-5 scale, adults 20+)
- **Approach:** Ordinal encoding (0-5): "Don't know" (9) → 0, keep 1-5 as-is
- **Rationale:** Tree-based models naturally evaluate all possible threshold splits on ordinal features. Preserves natural ordering while keeping it as a single feature (cleaner than one-hot encoding).

#### 3. Marital Status: `has_partner`, `had_partner`
**Source:** `DMDMARTZ`
- **Approach:** One-hot encoding (k-1 method)
  - `has_partner`: 1 if married/living with partner (code 1)
  - `had_partner`: 1 if widowed/divorced/separated (code 2)
  - Reference level: Never married (3) + Refused/Don't know
- **Rationale:** Nominal variable with no natural ordering. k-1 approach avoids multicollinearity while capturing all information.

#### 4. Gender: `is_female`
**Source:** `RIAGENDR`
- **Approach:** Binary encoding - `is_female` (1 if female, 0 if male)
- **Rationale:** Simple binary feature. Male serves as reference category.

#### 5. Blood Pressure: `systolic_bp`, `diastolic_bp`
**Source:** 3 readings each for systolic (`BPXOSY1-3`) and diastolic (`BPXODI1-3`)
- **Approach:** Row-wise mean across available (non-null) measurements
- **Rationale:** Standard clinical practice to reduce measurement variability. Averaging handles missing values automatically (pandas `.mean(axis=1)` ignores NaN).

#### 6. Smoking: `ever_smoker`, `is_current_smoker`
**Source:** `SMQ020` (ever smoked 100+ cigarettes), `SMQ040` (current status)
- **Approach:**
  - Clean: Refused (7) and Don't know (9) → Never smokers (2)
  - `ever_smoker`: Binary from cleaned SMQ020
  - `is_current_smoker`: 1 if ever smoked AND currently smoking (SMQ040 in [1,2])
- **Rationale:** Refused/don't know about 100+ cigarettes indicates minimal smoking history. Current smoker logic properly handles skip pattern (non-smokers skip SMQ040). Drops `SMD650` (cigarettes/day) due to 85% missingness.

#### 7. Alcohol: `drinking_frequency`, `drinks_per_day`, `binge_episodes_month`
**Source:** `ALQ121` (frequency), `ALQ130` (drinks/day), `ALQ170` (binge episodes)
- **Approach:**
  - Clean: Refused/Don't know codes → NaN (tree models handle natively)
  - Group ALQ121: 11 categories → 5 ordinal levels (0=Never → 4=Daily)
  - Keep ALQ130 and ALQ170 as continuous features
- **Rationale:** 
  - Grouping preserves dose-response relationship while improving efficiency
  - Ordinal encoding more efficient than one-hot for ordered categories
  - Complementary features: frequency + intensity + binge pattern = comprehensive alcohol profile

#### Metabolic History Features
**Source:** BPQ Questionnaire (Blood Pressure & Cholesterol)

| Feature | NHANES Variable | Transformation |
|---------|----------------|----------------|
| `told_high_bp` | BPQ020 | Binary: 1=Yes, 0=No. Refused/Don't know → NaN |
| `told_high_cholesterol` | BPQ080 | Binary: 1=Yes, 0=No. Refused/Don't know → NaN |

**Feature Availability Gate:** Both pass — these are patient self-report questions obtainable in a 5-minute pharmacy questionnaire.

**Cross-Cycle Note:** Both `BPQ020` and `BPQ080` have identical coding across cycles (1=Yes, 2=No, 7=Refused, 9=Don't know) and the same variable names in both raw files. `BPQ040A` (hypertension medication) was dropped in the 2021-2023 cycle and is NOT included.

#### Lab-Defined Targets
**Source:** GHB Lab File (Glycohemoglobin)

| Target | Definition | Notes |
|--------|-----------|-------|
| `lab_positive` | HbA1c ≥ 5.7% | ADA prediabetes threshold. NaN if no lab data. |
| `undiagnosed` | lab_positive=1 AND self_report=0 | Key thesis contribution — identifies undiagnosed cases. |

**Note:** Fasting glucose (LBXGLU) was excluded because it requires separate fasting subsample weights (WTSAF2YR) and has substantially lower coverage than HbA1c. `LBXGH` is dropped after target construction — it is NOT a model feature (not available at pharmacy inference time).

In [ ]:
# Apply common transformations to both datasets
print("Applying common transformations...")  # noqa: T201
df_2021 = apply_common_transformations(df_2021)
df_2017 = apply_common_transformations(df_2017)

print(f"✓ 2021-2023: {len(df_2021):,} rows after filtering")  # noqa: T201
print(f"✓ 2017-2020: {len(df_2017):,} rows after filtering")  # noqa: T201

## Physical Activity Transformations (Cycle-Specific)

The two cycles have different PA variable structures. We'll convert both to **weekly minutes** for consistency.

### 2021-2023 Approach:
- Calculate: `(frequency / every_X_days) × 7 × minutes_per_session`
- Convert unit (D/W/M/Y) to days, then normalize to weekly frequency

### 2017-2020 Approach:
- Calculate: `gate × days_per_week × minutes_per_session`
- Gate question (1=Yes, 2=No) filters who does activity
- Already in days per week, no unit conversion needed

### Transform 2021-2023 Physical Activity

**Transformation Rationale:**

**Goal:** Convert to **weekly minutes** for moderate and vigorous activity

**Source Variables:**
- `PAD790Q`: Frequency of moderate activity (e.g., "3")
- `PAD790U`: Unit (D/W/M/Y = Daily/Weekly/Monthly/Yearly)
- `PAD800`: Minutes per session
- `PAD810Q`, `PAD810U`, `PAD820`: Same structure for vigorous activity
- `PAD680`: Sedentary minutes per day

**Calculation Steps:**
1. **Clean sentinel values:** 7777 (refused), 9999 (don't know) → NaN
2. **Convert units to days:** D→1, W→7, M→30, Y→365
3. **Calculate weekly frequency:** `(frequency / every_X_days) × 7`
   - Example: "3 times per week" → `(3 / 7) × 7 = 3 times/week`
   - Example: "2 times per month" → `(2 / 30) × 7 = 0.47 times/week`
4. **Calculate weekly minutes:** `weekly_frequency × minutes_per_session`

**Rationale:** 
- Normalizes different reporting frequencies to a unified metric
- Weekly minutes enables direct comparison between cycles
- Tree models can handle NaN for missing/refused values naturally

In [ ]:
# Clean sentinel values
for col in ["PAD790Q", "PAD810Q", "PAD800", "PAD820", "PAD680"]:
    df_2021[col] = df_2021[col].replace([7777, 9999], pd.NA)  # type: ignore
    df_2021[col] = pd.to_numeric(df_2021[col], errors="coerce")  # type: ignore

# Convert unit to days mapping
unit_to_days = {"D": 1, "W": 7, "M": 30, "Y": 365}
df_2021["moderate_every_X_days"] = df_2021["PAD790U"].map(unit_to_days)  # type: ignore
df_2021["vigorous_every_X_days"] = df_2021["PAD810U"].map(unit_to_days)  # type: ignore

# Calculate weekly frequency: (frequency / every_X_days) * 7
df_2021["moderate_weekly_freq"] = (df_2021["PAD790Q"] / df_2021["moderate_every_X_days"]) * 7
df_2021["vigorous_weekly_freq"] = (df_2021["PAD810Q"] / df_2021["vigorous_every_X_days"]) * 7

# Calculate weekly minutes: weekly_freq * minutes_per_session
df_2021["moderate_minutes_per_week"] = df_2021["moderate_weekly_freq"] * df_2021["PAD800"]
df_2021["vigorous_minutes_per_week"] = df_2021["vigorous_weekly_freq"] * df_2021["PAD820"]

# Keep sedentary as-is (already in minutes per day)
df_2021["sedentary_minutes_per_day"] = df_2021["PAD680"]

# Drop original PA columns
df_2021 = df_2021.drop(columns=[
    "PAD790Q", "PAD790U", "PAD800", "PAD810Q", "PAD810U", "PAD820", "PAD680",
    "moderate_every_X_days", "vigorous_every_X_days", "moderate_weekly_freq", "vigorous_weekly_freq"
])

print("✓ 2021-2023 PA transformation complete")  # noqa: T201

### Transform 2017-2020 Physical Activity

**Transformation Rationale:**

**Goal:** Convert to **weekly minutes** for moderate and vigorous activity

**Source Variables:**
- `PAQ650`: Vigorous gate question (1=Yes, 2=No)
- `PAQ655`: Vigorous days per week (if gate=Yes)
- `PAD660`: Vigorous minutes per session
- `PAQ665`, `PAQ670`, `PAD675`: Same structure for moderate activity
- `PAD680`: Sedentary minutes per day

**Calculation Steps:**
1. **Clean sentinel values:** 7777 (refused), 9999 (don't know) → NaN
2. **Map gate questions:** 1 (Yes) → 1, 2 (No) → 0
3. **Calculate weekly minutes:** `gate × days_per_week × minutes_per_session`
   - If gate=No (0), result is 0 (no activity)
   - If gate=Yes (1), uses actual days and minutes

**Key Differences from 2021-2023:** 
- Uses **gate questions** (Yes/No screening) before asking frequency
- Days per week already provided (no unit conversion needed)
- Gate multiplier naturally handles non-participants (0 minutes)

**Rationale:** 
- Gate-based logic preserves NHANES skip patterns
- Still produces weekly minutes metric compatible with 2021-2023 cycle
- Simple multiplication handles both participants and non-participants correctly

In [ ]:
# Clean sentinel values
for col in ["PAQ650", "PAQ655", "PAD660", "PAQ665", "PAQ670", "PAD675", "PAD680"]:
    df_2017[col] = df_2017[col].replace([7777, 9999], pd.NA)  # type: ignore
    df_2017[col] = pd.to_numeric(df_2017[col], errors="coerce")  # type: ignore

# Map gate questions: 1=Yes -> 1, 2=No -> 0
df_2017["moderate_gate"] = df_2017["PAQ665"].map({1: 1, 2: 0})
df_2017["vigorous_gate"] = df_2017["PAQ650"].map({1: 1, 2: 0})

# Calculate weekly minutes: gate * days_per_week * minutes_per_session
df_2017["moderate_minutes_per_week"] = (
    df_2017["moderate_gate"] * df_2017["PAQ670"] * df_2017["PAD675"]
)
df_2017["vigorous_minutes_per_week"] = (
    df_2017["vigorous_gate"] * df_2017["PAQ655"] * df_2017["PAD660"]
)

# Keep sedentary as-is (already in minutes per day)
df_2017["sedentary_minutes_per_day"] = df_2017["PAD680"]

# Drop original PA columns
df_2017 = df_2017.drop(columns=[
    "PAQ650", "PAQ655", "PAD660", "PAQ665", "PAQ670", "PAD675", "PAD680",
    "moderate_gate", "vigorous_gate"
])

print("✓ 2017-2020 PA transformation complete")  # noqa: T201

## Final Cleanup & Standardization

In [ ]:
# Drop features excluded from v1: SEQN, Race, Income-to-poverty ratio
exclude_cols = ["SEQN", "RIDRETH3", "INDFMPIR", "DMDHHSIZ"]
df_2021 = df_2021.drop(columns=[col for col in exclude_cols if col in df_2021.columns])
df_2017 = df_2017.drop(columns=[col for col in exclude_cols if col in df_2017.columns])

# Standardize weight column name to "survey_weight"
df_2021 = df_2021.rename(columns={"WTMEC2YR": "survey_weight"})
df_2017 = df_2017.rename(columns={"WTMECPRP": "survey_weight"})

print("✓ Excluded features dropped")  # noqa: T201
print("✓ Weight columns standardized")  # noqa: T201

In [ ]:
# Verify both datasets have the same columns
cols_2021 = set(df_2021.columns)
cols_2017 = set(df_2017.columns)

print("Schema consistency check:")  # noqa: T201
print(f"  2021-2023 columns: {len(cols_2021)}")  # noqa: T201
print(f"  2017-2020 columns: {len(cols_2017)}")  # noqa: T201

# Check for differences
only_in_2021 = cols_2021 - cols_2017
only_in_2017 = cols_2017 - cols_2021

if only_in_2021:
    print(f"  ⚠ Only in 2021-2023: {only_in_2021}")  # noqa: T201
if only_in_2017:
    print(f"  ⚠ Only in 2017-2020: {only_in_2017}")  # noqa: T201
if not only_in_2021 and not only_in_2017:
    print("  ✓ Schemas match perfectly!")  # noqa: T201

print("\nNew feature coverage:")  # noqa: T201
for col in ["told_high_bp", "told_high_cholesterol", "lab_positive", "undiagnosed"]:
    for label, df in [("2021-2023", df_2021), ("2017-2020", df_2017)]:
        if col in df.columns:
            non_null = df[col].notna().sum()
            pct = non_null / len(df) * 100
            print(f"  {label} {col}: {non_null:,}/{len(df):,} non-null ({pct:.1f}%)")  # noqa: T201

In [ ]:
# Move target variables to the end
target_cols = ["has_diabetes_or_prediabetes", "lab_positive", "undiagnosed"]
for i, df_ref in enumerate([df_2021, df_2017]):
    feature_cols = [c for c in df_ref.columns if c not in target_cols]
    available_targets = [c for c in target_cols if c in df_ref.columns]
    reordered = df_ref[feature_cols + available_targets]
    if i == 0:
        df_2021 = reordered
    else:
        df_2017 = reordered

print("✓ Target variables moved to end")  # noqa: T201
print(f"  Targets: {available_targets}")  # noqa: T201

## Concatenate Both Cycles

In [ ]:
# Concatenate both cycles
df_combined = pd.concat([df_2021, df_2017], ignore_index=True)

print("Combined dataset:")  # noqa: T201
print(f"  Total rows: {len(df_combined):,}")  # noqa: T201
print(f"  Total columns: {len(df_combined.columns)}")  # noqa: T201
print(f"  2021-2023 rows: {(df_combined['cycle'] == '2021-2023').sum():,}")  # noqa: T201
print(f"  2017-2020 rows: {(df_combined['cycle'] == '2017-2020').sum():,}")  # noqa: T201

# Show distribution of target variable
print("\nSelf-report target distribution:")  # noqa: T201
print(df_combined["has_diabetes_or_prediabetes"].value_counts(dropna=False))  # noqa: T201

print("\nLab-defined target distribution:")  # noqa: T201
print(df_combined["lab_positive"].value_counts(dropna=False))  # noqa: T201

print("\nUndiagnosed distribution:")  # noqa: T201
print(df_combined["undiagnosed"].value_counts(dropna=False))  # noqa: T201

# Prevalence by cycle (validation check — should be within ±2-3pp)
print("\nPrevalence by cycle:")  # noqa: T201
for target in ["has_diabetes_or_prediabetes", "lab_positive", "undiagnosed"]:
    print(f"\n  {target}:")  # noqa: T201
    for cycle in ["2021-2023", "2017-2020"]:
        subset = df_combined[df_combined["cycle"] == cycle]
        valid = subset[target].dropna()
        if len(valid) > 0:
            prev = valid.mean() * 100
            print(f"    {cycle}: {prev:.1f}% (n={len(valid):,})")  # noqa: T201

In [ ]:
# Preview combined dataset
df_combined.head(10)

In [ ]:
# Show dataset info
df_combined.info()

## Save Processed Dataset

In [ ]:
# Save combined processed dataset
output_path = "../dataset/processed_data_combined_metabolic_history.csv"
df_combined.to_csv(output_path, sep=",", index=False)

print(f"✓ Saved combined dataset to: {output_path}")  # noqa: T201
print(f"  Rows: {len(df_combined):,}")  # noqa: T201
print(f"  Columns: {len(df_combined.columns)}")  # noqa: T201
print(f"  Features: {len(df_combined.columns) - 3} + 3 targets")  # noqa: T201
# -3 for: has_diabetes_or_prediabetes, lab_positive, undiagnosed